In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers,models,optimizers,callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import cv2


In [5]:
DataSet_Directory=r'C:\Users\abdul\OneDrive\Desktop\Food Quality Inspection\FRUIT-16K'
Image_Size=(224,224)
Batch_Size=32
Seed=4
Initial_Epochs=15
Fine_Tune_Epochs=10
Total_Epochs=Initial_Epochs+Fine_Tune_Epochs

Pseudo_Spectral_Channel=5
Learning_Rate=1e-4
Learning_Rate_FineTune=1e-5
Model_Save="Food_Quality_Inspection_Model.h5"

In [11]:
Training_DataGen=ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=(0.7,1.3),
    shear_range=0.08,
    zoom_range=0.08,
    horizontal_flip=True,
    fill_mode='reflect'
)

Training_Gen=Training_DataGen.flow_from_directory(
    DataSet_Directory,
    target_size=Image_Size,
    batch_size=Batch_Size,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=Seed
)

Validation_Gen=Training_DataGen.flow_from_directory(
    DataSet_Directory,
    target_size=Image_Size,
    batch_size=Batch_Size,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=Seed
)

Found 12800 images belonging to 2 classes.
Found 3200 images belonging to 2 classes.


In [12]:
Classes=Training_Gen.classes
Class_Name=list(Training_Gen.class_indices.keys())
print(f"Class Indices : {Training_Gen.class_indices}")

Class_Weights=compute_class_weight('balanced',classes=np.unique(Classes),y=Classes)
Class_Weights={i:w for i,w in enumerate(Class_Weights)}
print(f"Computed Class Weights : {Class_Weights}")

Class Indices : {'Fresh': 0, 'Spoiled': 1}
Computed Class Weights : {0: 1.0, 1: 1.0}


In [15]:
def Build_Model(img_size=Image_Size,pseudo_channels=Pseudo_Spectral_Channel):
    input=layers.Input(shape=(img_size[0],img_size[1],3),name='rgb_input')
    Base_Model=tf.keras.applications.MobileNetV2(
        input_shape=(img_size[0],img_size[1],3),
        include_top=False,
        weights='imagenet'
    )

    Base_Model.trainable=False
    X=Base_Model(input)
    X=layers.GlobalAveragePooling2D()(X)
    X=layers.BatchNormalization()(X)
    X=layers.Dense(256,activation='relu')(X)
    X=layers.Dropout(0.4)(X)
    RGB_Feat=layers.Dense(128,activation='relu',name='RGB_Feat')(X)

    Y=layers.Conv2D(32,(3,3),padding='same',activation='relu')(input)
    Y=layers.BatchNormalization()(Y)
    Y=layers.MaxPooling2D((2,2))(Y)
    Y=layers.Conv2D(64,(3,3),padding='same',activation='relu')(Y)
    Y=layers.BatchNormalization()(Y)
    Y=layers.MaxPooling2D((2,2))(Y)

    Pseudo=layers.Conv2D(pseudo_channels,(1,1),activation='linear',name='pseudo_spectral_maps')(Y)

    Z=layers.Conv2D(64,(3,3),padding='same',activation='relu')(Pseudo)
    Z=layers.GlobalAveragePooling2D()(Z)
    Z=layers.BatchNormalization()(Z)
    Pseudo_Feat=layers.Dense(64,activation='relu',name='pseudo_feat')(Z)

    Fused=layers.Concatenate(name='fusion')([RGB_Feat,Pseudo_Feat])
    Fused=layers.BatchNormalization()(Fused)
    Fused=layers.Dense(128,activation='relu')(Fused)
    Fused=layers.Dropout(0.5)(Fused)
    Fused=layers.Dense(64,activation='relu')(Fused)
    Out=layers.Dense(1,activation='sigmoid',name='output')(Fused)

    model=models.Model(inputs=input,outputs=Out,name='Hybrid_RGB_Pseudospec_Model')
    return model,Base_Model

In [16]:
Model,BackBone=Build_Model()
Model.summary()

Model: "Hybrid_RGB_Pseudospec_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ rgb_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        896 │ rgb_input[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ rgb_input[0][0]   │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pseudo_spectral_ma… │ (None, 56, 56, 5) │        325 │ max_pooling2d_1[… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1280)      │      5,120 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │      2,944 │ pseudo_spectral_… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    327,936 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ conv2d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ RGB_Feat (Dense)    │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pseudo_feat (Dense) │ (None, 64)        │      4,160 │ batch_normalizat

 Total params: 2,685,190 (10.24 MB)

 Trainable params: 423,942 (1.62 MB)

 Non-trainable params: 2,261,248 (8.63 MB)

In [20]:
Model.compile(
    optimizer=optimizers.Adam(learning_rate=Learning_Rate),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]

)

CheckPoint_CB=callbacks.ModelCheckpoint('best_hybrid_model.h5',monitor='val_auc',mode='max',save_best_only=True)
Reduce_LR_CB=callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=3,verbose=1)
EarlyStop_CB=callbacks.EarlyStopping(monitor='val_auc',patience=6,restore_best_weights=True)

History1=Model.fit(
    Training_Gen,
    epochs=Initial_Epochs,
    validation_data=Validation_Gen,
    class_weight=Class_Weights,
    callbacks=[CheckPoint_CB,Reduce_LR_CB,EarlyStop_CB]
)

c:\Users\abdul\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6533 - auc: 0.7082 - loss: 0.6525

400/400 ━━━━━━━━━━━━━━━━━━━━ 835s 2s/step - accuracy: 0.6536 - auc: 0.7086 - loss: 0.6521 - val_accuracy: 0.6594 - val_auc: 0.7357 - val_loss: 0.7835 - learning_rate: 1.0000e-04
Epoch 2/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9238 - auc: 0.9754 - loss: 0.2089

400/400 ━━━━━━━━━━━━━━━━━━━━ 523s 1s/step - accuracy: 0.9239 - auc: 0.9755 - loss: 0.2088 - val_accuracy: 0.6913 - val_auc: 0.8113 - val_loss: 0.8513 - learning_rate: 1.0000e-04
Epoch 3/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9572 - auc: 0.9914 - loss: 0.1185

400/400 ━━━━━━━━━━━━━━━━━━━━ 528s 1s/step - accuracy: 0.9573 - auc: 0.9914 - loss: 0.1184 - val_accuracy: 0.6963 - val_auc: 0.8145 - val_loss: 1.0296 - learning_rate: 1.0000e-04
Epoch 4/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9617 - auc: 0.9934 - loss: 0.0999


Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
400/400 ━━━━━━━━━━━━━━━━━━━━ 574s 1s/step - accuracy: 0.9617 - auc: 0.9934 - loss: 0.0999 - val_accuracy: 0.7259 - val_auc: 0.8461 - val_loss: 0.8977 - learning_rate: 1.0000e-04
Epoch 5/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 582s 1s/step - accuracy: 0.9715 - auc: 0.9958 - loss: 0.0774 - val_accuracy: 0.7216 - val_auc: 0.8345 - val_loss: 1.0098 - learning_rate: 5.0000e-05
Epoch 6/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 589s 1s/step - accuracy: 0.9755 - auc: 0.9967 - loss: 0.0683 - val_accuracy: 0.7166 - val_auc: 0.8355 - val_loss: 1.0244 - learning_rate: 5.0000e-05
Epoch 7/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9776 - auc: 0.9975 - loss: 0.0606
Epoch 7: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
400/400 ━━━━━━━━━━━━━━━━━━━━ 595s 1s/step - accuracy: 0.9776 - auc: 0.9975 - loss: 0.0606 - val_accuracy: 0.7069 - val_auc: 0.8182 - val_loss: 1.1168 - learning_rate: 5.0000e-05
Epoch 8/15
400

400/400 ━━━━━━━━━━━━━━━━━━━━ 600s 1s/step - accuracy: 0.9787 - auc: 0.9974 - loss: 0.0613 - val_accuracy: 0.7400 - val_auc: 0.8615 - val_loss: 0.8504 - learning_rate: 2.5000e-05
Epoch 9/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9825 - auc: 0.9975 - loss: 0.0537

400/400 ━━━━━━━━━━━━━━━━━━━━ 584s 1s/step - accuracy: 0.9825 - auc: 0.9975 - loss: 0.0537 - val_accuracy: 0.7456 - val_auc: 0.8665 - val_loss: 0.8675 - learning_rate: 2.5000e-05
Epoch 10/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9778 - auc: 0.9973 - loss: 0.0621


Epoch 10: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.
400/400 ━━━━━━━━━━━━━━━━━━━━ 580s 1s/step - accuracy: 0.9778 - auc: 0.9973 - loss: 0.0621 - val_accuracy: 0.7487 - val_auc: 0.8723 - val_loss: 0.8382 - learning_rate: 2.5000e-05
Epoch 11/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 590s 1s/step - accuracy: 0.9829 - auc: 0.9979 - loss: 0.0512 - val_accuracy: 0.7356 - val_auc: 0.8580 - val_loss: 0.8774 - learning_rate: 1.2500e-05
Epoch 12/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 582s 1s/step - accuracy: 0.9840 - auc: 0.9983 - loss: 0.0461 - val_accuracy: 0.7450 - val_auc: 0.8611 - val_loss: 0.8898 - learning_rate: 1.2500e-05
Epoch 13/15
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9827 - auc: 0.9983 - loss: 0.0486
Epoch 13: ReduceLROnPlateau reducing learning rate to 6.24999984211172e-06.
400/400 ━━━━━━━━━━━━━━━━━━━━ 580s 1s/step - accuracy: 0.9827 - auc: 0.9983 - loss: 0.0486 - val_accuracy: 0.7237 - val_auc: 0.8401 - val_loss: 0.9992 - learning_rate: 1.2500e-05
Epoch 14/1